# Teil 4 - Evaluation

In diesem Notebook bewerte ich das gelernte Modell aus `model.ipynb`. Es soll `co2` aus `year` und `energy_per_capita` vorhersagen. Damit die Bewertung fair bleibt, wird nur mit den Trainingsdaten gelernt. Die eigentliche Kontrolle passiert danach mit den Testdaten.

In [1]:
import csv
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

features = ["year", "energy_per_capita"]

with open("data/co2_clean_1000.csv", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

X = np.array([[float(row[column]) for column in features] for row in rows], dtype=float)
y = np.array([float(row["co2"]) for row in rows], dtype=float)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

model = DecisionTreeRegressor(
    max_depth=2,
    random_state=42,
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

print("Entscheidungsbaum mit fit() trainiert und für die Evaluation bereit.")
print(f"Trainingsdaten: {len(X_train)}")
print(f"Testdaten: {len(X_test)}")

Entscheidungsbaum mit fit() trainiert und für die Evaluation bereit.
Trainingsdaten: 800
Testdaten: 200


## 4.1 Aussagekräftige Felder

Beim Entscheidungsbaum kann man die Wichtigkeit der Felder über `feature_importances_` anschauen. Diese Werte zeigen, welche Eingabefelder dem Modell beim Aufteilen des Baums am meisten helfen. Ein höherer Wert bedeutet hier, dass das Feld für die Vorhersage wichtiger war.

In [2]:
importance = sorted(
    zip(features, model.feature_importances_),
    key=lambda item: item[1],
    reverse=True,
)

print("Feld                 Wichtigkeit")
for field, score in importance:
    print(f"{field:<20} {score:11.3f}")

Feld                 Wichtigkeit
energy_per_capita          0.964
year                       0.036


Das wichtigste Feld ist `energy_per_capita`. Das ist plausibel, weil Energieverbrauch stark mit CO2-Emissionen zusammenhängt. `year` ist weniger wichtig, hilft aber leicht beim historischen Verlauf. Das Modell arbeitet also vor allem mit dem Energieverbrauch und nutzt die Jahreszahl nur als kleinere Ergänzung.

## 4.2 Messmetrik

Für dieses Regressionsmodell berechne ich MSE, RMSE, MAE und R². Zusätzlich ist für die vereinfachte hohe/niedrige Einordnung die Accuracy wichtig, weil sie zeigt, wie viele Testfälle in der Wahrheitsmatrix richtig eingeordnet wurden. So sieht man sowohl die Genauigkeit der Zahlenwerte als auch die Einordnung in hohe oder niedrige Emissionen.

In [3]:
mse = mean_squared_error(y_test, predictions)
rmse = math.sqrt(mse)
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"MSE: {mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"R2: {r2:.3f}")

MSE: 9271.962
RMSE: 96.291
MAE: 57.306
R2: 0.270


## 4.3 Wahrheitsmatrix, Sensitivität und Spezifizität

Für die Wahrheitsmatrix mache ich aus der Regression eine einfache Ja-Nein-Frage: Liegt der CO2-Wert über dem Median der Trainingsdaten? Werte darüber gelten hier als hohe Emissionen. Dadurch kann ich zählen, wie oft das Modell hohe und niedrige Emissionen richtig oder falsch einordnet.

In [4]:
threshold = float(np.median(y_train))
actual_high = y_test >= threshold
predicted_high = predictions >= threshold

tn, fp, fn, tp = confusion_matrix(actual_high, predicted_high, labels=[False, True]).ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
precision = tp / (tp + fp)
accuracy = (tp + tn) / len(y_test)

print(f"Schwelle für hohe Emissionen: {threshold:.3f}")
print("Wahrheitsmatrix (Testdaten):")
print("                         Vorhergesagt niedrig  Vorhergesagt hoch")
print(f"Tatsächlich niedrig {tn:>24} {fp:>18}")
print(f"Tatsächlich hoch    {fn:>24} {tp:>18}")
print()
print(f"TP: {tp}")
print(f"TN: {tn}")
print(f"FP: {fp}")
print(f"FN: {fn}")
print(f"Sensitivität / Recall: {sensitivity:.3f}")
print(f"Spezifizität: {specificity:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Accuracy: {accuracy:.3f}")

Schwelle für hohe Emissionen: 28.557
Wahrheitsmatrix (Testdaten):
                         Vorhergesagt niedrig  Vorhergesagt hoch
Tatsächlich niedrig                       64                 30
Tatsächlich hoch                          10                 96

TP: 96
TN: 64
FP: 30
FN: 10
Sensitivität / Recall: 0.906
Spezifizität: 0.681
Precision: 0.762
Accuracy: 0.800


Die Wahrheitsmatrix steht oben direkt als Tabelle. `TP` sind korrekt erkannte hohe Emissionen, `TN` korrekt erkannte niedrige Emissionen. `FP` bedeutet, dass das Modell einen Wert zu hoch einschätzt. `FN` bedeutet, dass es hohe Emissionen übersieht.

In [5]:
plt.figure(figsize=(7, 5))
plt.scatter(y_test, predictions, alpha=0.7)
minimum = min(y_test.min(), predictions.min())
maximum = max(y_test.max(), predictions.max())
plt.plot([minimum, maximum], [minimum, maximum], color="red", linestyle="--")
plt.xlabel("Tatsächliche CO2-Werte")
plt.ylabel("Vorhergesagte CO2-Werte")
plt.title("Entscheidungsbaum: echte und vorhergesagte Werte")
plt.tight_layout()
plt.savefig("plots/model_predictions.png", dpi=160)
plt.show()

## 4.4 Zusammenfassung

Das Modell ordnet 80 Prozent der Testfälle richtig als hohe oder niedrige Emissionen ein. Für die grobe Einordnung funktioniert es also brauchbar. Die exakten CO2-Werte sind weniger genau, weil der Baum bewusst sehr einfach gehalten ist und nur wenige Gruppen bildet. Bessere Werte wären möglich, wenn man mehr Felder oder einen komplexeren Baum verwenden würde.

![Vergleich tatsächliche und vorhergesagte Werte](plots/model_predictions.png)